---
title: Keecas Quarto Example
toc: true
format:
    html:
        include-in-header:
            # this is needed for equation numbering and labeling in HTML output             
            - text: | 
                <script>
                MathJax = { tex: { tags: 'ams' } };
                </script>
            # this get rid of unnecessary vertical scrollbars
            - text: |
                <style>
                .math.display {
                  max-width: 100%;
                  padding-right: 3em;
                }
                </style>
    pdf: default
echo: false
keep-tex: false
---

# How to use keecas in a Quarto document

This example demonstrates all the key features of `keecas` for symbolic and units-aware calculations in a Quarto document.

In [1]:
# for quick start import *
# from keecas import *

# or explicit import (this is about all you need)
from keecas import check, config, pc, show_eqn, symbols, u

In [2]:
config.language.language = "en"

::: {.callout-note}

If the rendering engine is `KaTeX` (i.e. Jupyter notebook), the label command `\label{}` will result in a `ParseError`. The latex code will work when rendering by `quarto render`, but since you may want to see the result before rendering the document, an option to disable the `\label{}` command is provided.

When rendering with `quarto` you can have two config:

- if `qmd` files, then everything is fine
- if `ipynb` files, then pass the `--execute` flag to `quarto render` to rerender the notebook

:::


In [3]:
# | eval: false

# this option will not be set at rendering time (dev-mode)
config.display.katex = True  # disable `\label command`
config.display.print_label = True  # print interpolated labels for easy reference (e.g. copy and paste)

In [4]:
# set a prefix for the latex equation
config.latex.eq_prefix = r"eq-QUARTO_EXAMPLE-"


# Initialize notebook-global dictionaries for persistence across cells
params = {}  # Global parameters that persist throughout the notebook
eqn = {}  # Global expressions that persist throughout the notebook

## Basic Symbolic Math with Units

Define symbols and calculate basic engineering quantities:

In [5]:
# Define symbols
F, A, sigma = symbols("F, A, sigma")

# Parameters with units
_p = {
    F: 10 * u.kN,
    A: 50 * u.cm**2,
}
params.update(_p)  # Save to global params

# Symbolic expressions
_e = {sigma: "F / A" | pc.parse_expr}
eqn.update(_e)  # Save to global expressions

# Evaluated results
_v = {k: v | pc.subs(_e | _p) | pc.convert_to([u.MPa]) | pc.N for k, v in _e.items()}

# Descriptions
_d = {F: "applied force", A: "cross-sectional area", sigma: "normal stress"}

# Labels
_l = {k: str(k) for k in _d.keys()}

show_eqn(
    [_p | _e, _v, _d],
    float_format="{:.2f}",
    label=_l,
    # debug=True,
)

# the labels will be printed for easy reference, only in dev-mode (not rendered by quarto)

F: F
A: A
sigma: sigma


<IPython.core.display.Latex object>

## Pipe Commands Demonstration {#sec-pipe}

Using pipe operators for functional composition:

In [6]:
# Chain operations using pipe commands
x, y, d = symbols("x, y, d")

# Parameters
_p = {
    x: 3 * u.m,
    y: 4 * u.m,
}

# Symbolic expressions
_e = {d: "x^2 + y^2" | pc.parse_expr}

# Evaluated results
_v = {k: v | pc.subs(_e | _p) | pc.convert_to([u.m]) | pc.N for k, v in _e.items()}

show_eqn([_p | _e, _v])

<IPython.core.display.Latex object>

## Different LaTeX Environments

### Cases Environment

In [7]:
# Material properties
E_steel, E_concrete = symbols("E_{steel}, E_{concrete}")

_p = {
    E_steel: 200 * u.GPa,
    E_concrete: 30 * u.GPa,
}


_d = {E_steel: "steel elastic modulus", E_concrete: "concrete elastic modulus"}



### Equation Environment with Labels {#sec-beam-calc}

Calculate beam deflection with cross-references:

In [8]:
# Beam calculation
q, L, E, I, delta = symbols("q, L, E, I, delta")

_p = {
    q: 5 * u.kN / u.m,
    L: 8 * u.m,
    E: 200 * u.GPa,
    I: 8360 * u.cm**4,
}
params.update(_p)  # Save to global params

# Deflection formula
_e = {delta: "5 * q * L^4 / (384 * E * I)" | pc.parse_expr}
eqn.update(_e)  # Save to global expressions

_v = {k: v | pc.subs(_e | _p) | pc.convert_to([u.mm]) | pc.N for k, v in _e.items()}

_l = {k: str(k) for k in _e.keys()}

show_eqn([_e, _v], label=_l, float_format="{:.2f}")

delta: delta


<IPython.core.display.Latex object>

::: {.callout-note}

labels emitted by `show_eqn` are latex labels, therefore they need to be referenced with `\eqref{}` or `\ref{}` latex commands.

@eq-QUARTO_EXAMPLE-delta will not work

:::

The deflection calculated in \ref{eq-QUARTO_EXAMPLE-delta} shows acceptable values for serviceability.

## Verification Function

Using the `check` function for design checks:

In [9]:
# Design checks
sigma_Sd, tau_Sd, sigma_Rd, tau_Rd = symbols(
    r"\sigma_{Sd}, \tau_{Sd}, \sigma_{Rd}, \tau_{Rd}",
)

_p = {
    sigma_Sd: 20 * u.MPa,
    tau_Sd: 15 * u.MPa,
    sigma_Rd: 250 * u.MPa,
    tau_Rd: 75 * u.MPa,
}

# expressions to check
_expr = [
    sigma_Sd / sigma_Rd,
    tau_Sd / tau_Rd,
]

# evaluate expresions
_v = {k: k | pc.subs(_e | _p) | pc.N for k in _expr}

# check if expressions are less than 1
_c = {k: check(v, 1.0) for k, v in _v.items()}

# specify float format only for the check values
_ff = {k: [None, "{:.3f}", None] for k in _c.keys()}

show_eqn([_p | _v, _c], float_format=_ff)

<IPython.core.display.Latex object>

The type of comparison in the `check` function can be specified, as well the value to be specified against.

In [10]:
a, b, c, d, f = symbols("a, b, c, d, f")

from sympy import Eq, Ge, Gt, Le, Lt

_expr = {
    a: (3, Le, 1),
    b: (4, Gt, 2),
    c: (5, Lt, 3),
    d: (6, Ge, 4),
    f: (7, Eq, 5),
}

_c = {k: check(lhs=v[0], test=v[1], rhs=v[2]) for k, v in _expr.items()}

show_eqn(
    [
        {k: v[0] for k, v in _expr.items()},
        _c,
    ],
)

<IPython.core.display.Latex object>

## Subs get ordered

Using `pc.subs` will automatically order the substitutions pairs topologically, so you don't have to order them yourself (default `sympy` behavior). For example let's define some mappings in any order:

In [11]:
any_order_params = {
    x: y**2,
    y: a+1,
    a: 3,
}


display(show_eqn(_e, environment="cases"))

<IPython.core.display.Latex object>

If we use the default behavior of sympy's `subs` method (unsorted substitutions) we get:

In [12]:
# `pc.subs(any_order_params, sorted=False)` it's equivalent to use the sympy subs method
_v = {
    lhs: rhs | pc.subs(any_order_params, sorted=False) for lhs, rhs in any_order_params.items()
}

_d = {
    x: "partially evaluated",
}

show_eqn(
    [_v, _d],
    environment="cases",
)

<IPython.core.display.Latex object>

While the default behavior of `pc.subs` is to sort the substitutions:

In [13]:
_v = {
    lhs: rhs | pc.subs(any_order_params) for lhs, rhs in any_order_params.items()
}

_d = {
    x: "fully evaluated",
}

show_eqn(
    [_v, _d],
    environment="cases",
)

<IPython.core.display.Latex object>

::: {.callout-note}

`pc.subs` also take care of sympyfing the object before subsituting. In the above example $a$ is mapped to `3`, an `int`, therefore during dict comprehension the `int` is piped to `pc.subs`. If you would have used the `sympy` subs method directly, you would have get an error because the type `int` does not have `subs` method (the corrected dict comprehension you should look like  `S(rhs).subs(...)`).

:::

## Check Function Templates

The `check` function supports customizable templates for different visual styles:

In [14]:
# Default template behavior
result_default = check(0.8, 1.0)

# Using named template sets
result_boxed = check(0.8, 1.0, template="boxed")
result_minimal = check(0.8, 1.0, template="minimal")

# Demonstration of different template styles
template_demo = {
    "Default": result_default,
    "Boxed": result_boxed,
    "Minimal": result_minimal,
}

show_eqn(template_demo, col_wrap=[None, ":", "&"], environment="align")

<IPython.core.display.Latex object>

In [15]:
# Complex expression with multiple substitutions
a, b, c, d = symbols("a, b, c, d")


_p = {
    a: 3,
    b: 4,
}
params.update(_p)

_e = {
    d: "sqrt(a^2 + b^2) / c" | pc.parse_expr,
    c: "a*b" | pc.parse_expr,
}
eqn.update(_e)

_v = {
    k: v | pc.subs(eqn|params) | pc.N for k, v in _e.items()
}

show_eqn([_p|_e, _v], float_format="{:.3f}")

<IPython.core.display.Latex object>

### Custom Templates

You can also use completely custom templates:

In [16]:
# Custom template examples (emoji may not render in latex)
custom_success = r"✅ ${symbol}{rhs}$ \textbf{{PASS}}"
custom_failure = r"❌ ${symbol}{rhs}$ \textbf{{FAIL}}"

# Test both success and failure cases
ratio_ok = 0.8  # Should pass
ratio_fail = 1.2  # Should fail

check_ok = check(ratio_ok, 1.0,
                success_template=custom_success,
                failure_template=custom_failure)

check_fail = check(ratio_fail, 1.0,
                  success_template=custom_success,
                  failure_template=custom_failure)

custom_demo = {
    "Pass": check_ok,
    "Fail": check_fail,
}

show_eqn(custom_demo, col_wrap=[None, ":", "&"])

<IPython.core.display.Latex object>

## Summary

This example demonstrated:

- Basic symbolic math with units
- Pipe command usage (see @sec-pipe)
- Different LaTeX environments (align, cases, equation)
- Label and cross-reference functionality
- Dataframe operations
- Verification functions
- Complex symbolic manipulations

All calculations in @sec-beam-calc show that keecas provides a powerful interface for engineering calculations in Quarto documents.

### Template Configuration

Templates can also be configured globally via TOML config files:

```toml
[check_templates.template_sets.minimal]
success = '${symbol}{rhs} \,\textcolor{{green}}{{\checkmark}}$'
failure = '${symbol}{rhs} \,\textcolor{{red}}{{\times}}$'
```

This allows you to set project-wide or user-wide styling for all check functions.

## Test using a `QuartoMarkdownBase64`

In [17]:
a, b, c = symbols(r"a b c")

_p = {
    a: 5 * u.m,
    b: 12 * u.m,
    c: a+b,
}

import base64

encode = lambda s: base64.b64encode(s.encode()).decode()

_d = {
    # a: fr"\QuartoMarkdownBase64{{{encode("length **a**")}}}",
    b: "length b",
    c: "total length c",
}

show_eqn(
    [_p, _d],
    environment={
        'separator': ' & ',
        'line_separator': r' \\' + '\n',
        'supports_multiple_labels': True,
        'outer_environment': 'align',
        'inner_environment': None,
        'inner_prefix': '',
        'inner_suffix': '',
        'outer_prefix': '```{=latex}\n',
        'outer_suffix': '\n```',
        'label_position': 'outer',
    },
    debug=True,
)



```{=latex}
\begin{align}
a  &  = 5{\,}\text{m}  &     \\[8pt]
b  &  = 12{\,}\text{m}  &  \quad\text{length b}  \\[8pt]
c  &  = a + b  &  \quad\text{total length c} 
\end{align}
```


<IPython.core.display.Latex object>